In [ ]:
import pandas as pd

roll_no = "1024170192"

fixed_entries = [
    {"question": "what is the annual fee", "answer": "The annual fee is Rs 500.", "keywords": "fee cost price charge", "category": "billing"},
    {"question": "how to reset password", "answer": "Go to Settings > Reset Password.", "keywords": "password reset login", "category": "account"},
    {"question": "what are your working hours", "answer": "We are open 9 AM to 5 PM.", "keywords": "hours timing open time", "category": "general"},
    {"question": "how can i pay the fee", "answer": "You can pay via UPI, card, or net banking.", "keywords": "pay payment upi fee", "category": "billing"}
]

categories = ["billing", "account", "general"]

digits = [int(roll_no[-2]), int(roll_no[-1])]

personalized = []

for d in digits:
    category = categories[d % 3]

    if category == "billing":
        question = "how can i check my payment status"
        answer = "You can check your payment status from the billing section."
        keywords = "payment status billing"
    elif category == "account":
        question = "how do i update my mobile number"
        answer = "You can update your mobile number from account settings."
        keywords = "mobile number update"
    else:
        question = "where can i find general information"
        answer = "You can find general information on the help page."
        keywords = "information help general"

    personalized.append({
        "question": question,
        "answer": answer,
        "keywords": keywords,
        "category": category
    })

df = pd.DataFrame(fixed_entries + personalized)

print(df) 

                               question  \
0                what is the annual fee   
1                 how to reset password   
2           what are your working hours   
3                 how can i pay the fee   
4     how can i check my payment status   
5  where can i find general information   

                                              answer  \
0                          The annual fee is Rs 500.   
1                   Go to Settings > Reset Password.   
2                          We are open 9 AM to 5 PM.   
3         You can pay via UPI, card, or net banking.   
4  You can check your payment status from the bil...   
5  You can find general information on the help p...   

                   keywords category  
0     fee cost price charge  billing  
1      password reset login  account  
2    hours timing open time  general  
3       pay payment upi fee  billing  
4    payment status billing  billing  
5  information help general  general  


In [6]:
def score_query(query, df):
    query_words = set(query.lower().split())
    results = []

    for i, row in df.iterrows():
        keywords = set(row["keywords"].lower().split())
        score = len(query_words & keywords)

        if score > 0:
            results.append({
                "question": row["question"],
                "answer": row["answer"],
                "category": row["category"],
                "confidence": score
            })

    results.sort(key=lambda x: x["confidence"], reverse=True)
    return results

query = input("Enter your query: ")
print(score_query(query, df))

[{'question': 'how can i check my payment status', 'answer': 'You can check your payment status from the billing section.', 'category': 'billing', 'confidence': 1}]


In [7]:
def same_category(category_name, df):
    return df[df["category"] == category_name]

category_name = df.iloc[-1]["category"]

print(same_category(category_name, df))

                               question  \
2           what are your working hours   
5  where can i find general information   

                                              answer  \
2                          We are open 9 AM to 5 PM.   
5  You can find general information on the help p...   

                   keywords category  
2    hours timing open time  general  
5  information help general  general  


In [8]:
index = 0

new_keyword = input("Enter a new keyword: ")

df.loc[index, "keywords"] = df.loc[index, "keywords"] + " " + new_keyword

df.to_csv(roll_no + "_faq_data.csv", index=False)

print(df)

                               question  \
0                what is the annual fee   
1                 how to reset password   
2           what are your working hours   
3                 how can i pay the fee   
4     how can i check my payment status   
5  where can i find general information   

                                              answer  \
0                          The annual fee is Rs 500.   
1                   Go to Settings > Reset Password.   
2                          We are open 9 AM to 5 PM.   
3         You can pay via UPI, card, or net banking.   
4  You can check your payment status from the bil...   
5  You can find general information on the help p...   

                    keywords category  
0  fee cost price charge hey  billing  
1       password reset login  account  
2     hours timing open time  general  
3        pay payment upi fee  billing  
4     payment status billing  billing  
5   information help general  general  


In [9]:
print(df.groupby("category").size())

category
account    1
billing    3
general    2
dtype: int64


In [10]:
def score_query_tie(query, df):
    query_words = set(query.lower().split())
    results = []

    for i, row in df.iterrows():
        keywords = set(row["keywords"].lower().split())
        score = len(query_words & keywords)

        results.append({
            "question": row["question"],
            "answer": row["answer"],
            "category": row["category"],
            "confidence": score
        })

    results.sort(key=lambda x: x["confidence"], reverse=True)

    highest = results[0]["confidence"]

    if highest > 0:
        for result in results:
            if result["confidence"] == highest:
                print(result)
    else:
        print("No match found")

print("Tie query:")
score_query_tie("fee", df)

print("Non-tie query:")
score_query_tie("password", df)

Tie query:
{'question': 'what is the annual fee', 'answer': 'The annual fee is Rs 500.', 'category': 'billing', 'confidence': 1}
{'question': 'how can i pay the fee', 'answer': 'You can pay via UPI, card, or net banking.', 'category': 'billing', 'confidence': 1}
Non-tie query:
{'question': 'how to reset password', 'answer': 'Go to Settings > Reset Password.', 'category': 'account', 'confidence': 1}
